# 1 · Your first no-show model 🩺

**What you practise (everything from the last four days, on a new problem):**
- Framing: target, **prediction moment**, and which mistake hurts more (Day 1).
- Spotting **leaks**: columns we don't know yet when we predict (Day 4).
- A **baseline** to beat, and why accuracy alone can fool you.
- Turning raw dates into features, and a **split by date**.
- One recipe (`Pipeline`), three models: **logistic regression, KNN, decision tree** (Days 1–3).
- Reading **confusion matrices**, spotting **overfitting**, and asking what the model relies on.

## 🩺 The case: PraxisNow

**PraxisNow** is a booking app for a group of medical practices in five German cities. Patients book a slot in the app, on the website or by phone.

**What already happens today:** 24 hours before every appointment, each patient who agreed to receive SMS (about 2 in 3) gets a reminder: *"Your appointment is tomorrow at 10:00. Reply YES to confirm."*

**The problem:** even so, about **1 in 5 booked appointments ends with an empty chair**: the patient doesn't come and doesn't cancel. Every empty slot costs the practice about **€60** (the doctor is paid anyway), and another patient could have had that slot.

**Lena**, Head of Operations, asks:

*"When an appointment is **booked**, can we tell how likely it is that the patient won't show up?"*

With that score she wants to decide three things. You'll make these decisions **tomorrow** (notebook 2):

| | What it means | Cost | If the model is wrong… |
|---|---|---|---|
| ☎️ **Call** | Someone at the front desk phones the patient a few days before: *"Do you still need this appointment?"* | staff time: only **~25 calls a week** | one wasted call |
| 📅 **Double-book** | Give the same slot to a second patient from the waiting list, expecting the first one not to come | nothing upfront | if **both** come, one waits ~45 min |
| 📱 **SMS** *(already running)* | Finance asks: *"Could we save money by texting only the risky patients?"* | €0.10 per SMS | – |

**Dr. Okafor**, the medical director, adds: *"Whatever you build, I want to understand why a patient gets flagged."*

> 📌 The data is **synthetic** (made for this course) but realistic: 6,228 appointments from January to October 2026. No real patients.

> **How this notebook works**
> - Run the cells **from top to bottom** with `Shift + Enter`.
> - ▶️ **Run** cells work as they are.
> - ✏️ **Your turn** cells contain `___` (three underscores). Replace each `___` with your answer, then run the cell.
> - ✅ **Check** cells tell you whether you got it right. If a check fails, read the hint, fix the cell above and run both again.
> - 🧠 **Think** cells ask a PM question. Double-click the answer cell below it and write 1–3 sentences.
> - If you see `NameError: name '___' is not defined`, a blank is still empty.
> - Stuck? The `solutions/` folder has the same notebook, solved.

🧠 **Think (before any code):** Which mistake is worse: a patient flagged as "won't come" who **comes**, or a patient flagged as "will come" who **doesn't**? *Hint: think about the call and the double-booking separately.*

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 0 · Load the tools and the data

In [ ]:
# ▶️ Run: libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.base import clone

appointments = pd.read_csv("data/praxisnow_appointments.csv", parse_dates=["booked_at", "appointment_at"])
appointments.head()

| column | meaning |
|---|---|
| `appointment_id`, `patient_id` | IDs (a patient can have several appointments) |
| `booked_at` | the day the appointment was booked |
| `appointment_at` | date and time of the appointment |
| `clinic_city` | Berlin, Hamburg, Munich, Cologne, Leipzig |
| `specialty` | general_practice, dermatology, dentistry, physiotherapy, orthopaedics |
| `booking_channel` | app, website, phone |
| `age` | patient's age in years |
| `insurance` | `public` or `private` health insurance |
| `distance_km` | distance from home to the practice (empty = address not shared) |
| `sms_opt_in` | 1 = the patient agreed to receive SMS reminders |
| `prior_appointments` | the patient's appointments that were **already over on the booking day** |
| `prior_no_shows` | how many of those they didn't show up to |
| `reminder_reply` | reply to the reminder SMS sent **24 h before** the appointment: `confirmed` / `no_reply` (empty = no SMS) |
| `wait_time_min` | minutes the patient waited in the waiting room |
| **`no_show`** | **target**: 1 = the patient didn't come and didn't cancel |

## Step 1 · Size up the problem

In [ ]:
# ✏️ Your turn: how many appointments, and what share were no-shows?
# Hint: `no_show` is 0/1, so its average IS the share of 1s.
n_appointments = appointments.___[0]
no_show_share = appointments["no_show"].___()
print("Appointments:", n_appointments)
print("No-show share:", round(no_show_share, 3))

In [ ]:
# ✅ Check
assert n_appointments == 6228, "Use appointments.shape[0] (rows come first)"
assert abs(no_show_share - 0.2099) < 0.0005, "Use .mean() on the no_show column"
print("✅ Correct: 21% no-shows, about 1 in 5. That's 1,307 empty chairs in ten months, about €78,420.")

## Step 2 · Which columns may the model use?

**The snapshot rule (Day 4):** a feature may only contain what we know **at the moment we predict**. Here that moment is **when the appointment is booked**.

Two columns break that rule. One of them is easy to catch with a single line:

In [ ]:
# ▶️ Run: share of EMPTY values per column, for patients who came (0) vs no-shows (1)
empty_share = appointments.drop(columns="no_show").isna().groupby(appointments["no_show"]).mean().T
empty_share.columns = ["came (0)", "no-show (1)"]
empty_share.round(2)

Look at the `wait_time_min` row: it's empty for **every** no-show and for **none** of the patients who came. Of course: you can only wait in the waiting room if you showed up. A model would learn "empty = no-show" and look perfect, and be useless on the day of booking.

The second one is quieter: `reminder_reply` is a real signal (patients who confirmed almost always come), but the SMS goes out **24 hours before** the appointment, often weeks **after** booking.

In [ ]:
# ✏️ Your turn: list the two columns that break the snapshot rule (as strings)
leaky_columns = [___, ___]

In [ ]:
# ✅ Check
assert set(leaky_columns) == {"wait_time_min", "reminder_reply"}, "One is only filled if the patient came; one arrives 24 h before the appointment"
print("✅ Correct. Both go out. IDs go out too: they name a row, they don't describe it.")

> 💡 What about `prior_no_shows`? It's allowed: it only counts appointments that were **already over on the booking day**. An appointment that was still in the future when this one was booked isn't counted, even if the patient later skipped it.
>
> 📌 One simplification: the table only contains appointments that were **not cancelled** in advance. At booking we can't know that yet, so a real system would also score bookings that later get cancelled.

🧠 **Think:** `reminder_reply` isn't *wrong* data. When does it become available, and is there a **different decision**, made later, where it would be a fair feature?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 3 · The baseline to beat

Before any model: what if we simply predict **"everyone comes"** (always 0)?

In [ ]:
# ✏️ Your turn: accuracy and recall of "everyone comes"
# Hint: use the variable no_show_share from Step 1. Predicting 0 for everyone is right for every patient who came.
baseline_accuracy = 1 - ___
baseline_recall = ___   # how many no-shows does "everyone comes" catch?
print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Baseline recall:  ", baseline_recall)

In [ ]:
# ✅ Check
assert abs(baseline_accuracy - 0.7901) < 0.0005, "1 minus the no-show share"
assert baseline_recall == 0, "It never predicts a no-show, so it catches none"
print("✅ Correct: 79% accuracy without any model, and it catches ZERO no-shows. 🚩 Red flag: 'accuracy' alone means little when one class is rare.")

## Step 4 · Turn dates into features

A model can't use a raw date. But **how far ahead** the patient booked, the **weekday** and the **hour** might matter.

In [ ]:
# ✏️ Your turn: build three features from the two date columns
# Hint: .dt.days turns a time difference into a number of days; .dt.hour takes the hour of a date-time.
appointments["lead_days"] = (appointments["appointment_at"].dt.normalize() - appointments["booked_at"]).dt.___
appointments["weekday"] = appointments["appointment_at"].dt.day_name()
appointments["hour"] = appointments["appointment_at"].dt.___
appointments[["booked_at", "appointment_at", "lead_days", "weekday", "hour"]].head()

In [ ]:
# ✅ Check
assert appointments["lead_days"].min() == 0 and appointments["lead_days"].max() == 120, "Use .dt.days"
assert set(appointments["hour"]) == set(range(8, 19)), "Use .dt.hour"
print("✅ Correct: lead time goes from 0 (same day) to 120 days.")

In [ ]:
# ▶️ Run: no-show share by how far ahead the appointment was booked
lead_group = pd.cut(appointments["lead_days"], [-1, 0, 2, 7, 14, 30, 120],
                    labels=["same day", "1-2 days", "3-7 days", "1-2 weeks", "2-4 weeks", "1-4 months"])
by_lead = appointments.groupby(lead_group, observed=True)["no_show"].agg(["mean", "size"])
print(by_lead.round(3))
by_lead["mean"].plot.bar(color="#3b7dd8", rot=0, figsize=(8, 3.5), title="No-show share by lead time")
plt.ylabel("share of no-shows")
plt.show()

🧠 **Think:** Patients who book for the same day almost always come (6% no-shows); patients who book one to four months ahead often don't (45%). What could PraxisNow change **in the product** with this insight alone, without any model?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 5 · Split by date: train on the past, test on the "future"

In real use the model always predicts **upcoming** appointments using what it learned from **past** ones. So we split the same way:
- **Train:** appointments **before 1 August 2026** (January–July).
- **Test:** appointments from **1 August** on (August–October), our "future".

In [ ]:
# ✏️ Your turn: complete the two conditions (< or >=)
features = ["lead_days", "weekday", "hour", "clinic_city", "specialty", "booking_channel",
            "age", "insurance", "distance_km", "sms_opt_in", "prior_appointments", "prior_no_shows"]
cutoff = "2026-08-01"
is_train = appointments["appointment_at"] ___ cutoff
is_test = appointments["appointment_at"] ___ cutoff

X_train, y_train = appointments.loc[is_train, features], appointments.loc[is_train, "no_show"]
X_test, y_test = appointments.loc[is_test, features], appointments.loc[is_test, "no_show"]
print("Train:", X_train.shape, " no-show share", round(y_train.mean(), 3))
print("Test: ", X_test.shape, " no-show share", round(y_test.mean(), 3))

In [ ]:
# ✅ Check
assert len(X_train) == 4354 and len(X_test) == 1874, "Train = before the cutoff (<), test = from the cutoff on (>=)"
assert not set(leaky_columns) & set(X_train.columns)
print("✅ Correct: 4,354 past appointments to learn from, 1,874 'future' ones to test on.")
print("🏁 M1 reached: an honest table (no leaks, date features, split by time).")

> 📌 **A simplification, said out loud:** 216 of the test appointments were *booked* in July, before the cut-off. A model launched on 1 August could only score bookings made from then on. Splitting by appointment date is still a fair test for learning: the model only learns from appointments that were over by 1 August, and is tested on appointments it has never seen.

🧠 **Think:** Why split by date instead of picking test rows at random? What could a random split let the model "see" that it will never see in real life?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 6 · One recipe for every model

Same idea as Day 4: **numbers** get gaps filled (median + a "was missing" flag) and are scaled; **categories** are one-hot encoded, and unknown categories are ignored instead of crashing.

In [ ]:
# ✏️ Your turn: complete the recipe
numeric = ["lead_days", "hour", "age", "distance_km", "sms_opt_in", "prior_appointments", "prior_no_shows"]
categorical = ["weekday", "clinic_city", "specialty", "booking_channel", "insurance"]

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy=___, add_indicator=True)),
                      ("scale", ___)]), numeric),
    ("cat", OneHotEncoder(handle_unknown=___, sparse_output=False), categorical),
])
X_train_ready = prep.fit_transform(X_train)
print("Ready-to-learn table:", X_train_ready.shape)

In [ ]:
# ✅ Check
assert X_train_ready.shape == (4354, 28), "median · StandardScaler() · 'ignore'"
print("✅ Correct: 12 raw columns became 28 numbers (7 numeric + 1 'distance was missing' flag + 20 one-hot columns).")

## Step 7 · Three models, one fair comparison

The helper below puts **the same recipe** in front of any model, fits it on the training rows only, and scores it on train and test. The default rule is: **flag as no-show if the score is ≥ 0.5**.

In [ ]:
# ▶️ Run: the helper
def evaluate(name, model):
    pipe = Pipeline([("prep", clone(prep)), ("model", model)]).fit(X_train, y_train)
    pred_train, pred_test = pipe.predict(X_train), pipe.predict(X_test)
    score_test = pipe.predict_proba(X_test)[:, 1]
    row = {"model": name,
           "train accuracy": accuracy_score(y_train, pred_train),
           "test accuracy": accuracy_score(y_test, pred_test),
           "recall": recall_score(y_test, pred_test),
           "precision": precision_score(y_test, pred_test, zero_division=0),
           "F1": f1_score(y_test, pred_test),
           "test AUC": roc_auc_score(y_test, score_test)}
    return pipe, row

In [ ]:
# ✏️ Your turn: fill in the two missing model classes
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "KNN (k=15)": ___(n_neighbors=15),
    "Decision tree": ___(random_state=42),
}
fitted, rows = {}, [{"model": "Baseline: everyone comes", "train accuracy": 1 - y_train.mean(),
                     "test accuracy": 1 - y_test.mean(), "recall": 0.0, "precision": 0.0, "F1": 0.0, "test AUC": 0.5}]
for name, model in models.items():
    fitted[name], row = evaluate(name, model)
    rows.append(row)
results = pd.DataFrame(rows).set_index("model").round(3)
results

In [ ]:
# ✅ Check
assert abs(results.loc["Logistic regression", "test AUC"] - 0.782) < 0.002
assert results.loc["Decision tree", "train accuracy"] == 1.0
print("✅ Three models, one recipe, one table.")

🧠 **Think:** Logistic regression's test accuracy is **0.818**, only about 2 points above "everyone comes" (**0.797**). Is the model nearly useless? Look at **recall**, **precision** and **test AUC** before you answer.

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 8 · Read the confusion matrices

In [ ]:
# ▶️ Run: one confusion matrix per model (test set, threshold 0.5)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, pipe) in zip(axes, fitted.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test), ax=ax, colorbar=False,
                                            display_labels=["came", "no-show"])
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# ✏️ Your turn: pick the right cells of logistic regression's matrix
# Rows = what really happened (0 came, 1 no-show); columns = what the model said.
cm_lr = confusion_matrix(y_test, fitted["Logistic regression"].predict(X_test))
missed_no_shows = cm_lr[___, ___]       # really no-show, model said "came"
false_alarms = cm_lr[___, ___]          # really came, model said "no-show"
print("Missed no-shows:", missed_no_shows, " False alarms:", false_alarms)

In [ ]:
# ✅ Check
assert missed_no_shows == 272 and false_alarms == 70, "Row = truth, column = prediction"
print(f"✅ Correct: logistic regression misses {missed_no_shows} of {y_test.sum()} no-shows and raises {false_alarms} false alarms.")
print("🏁 M2 reached: three models compared against a baseline.")

🧠 **Think:** At the default threshold of 0.5, logistic regression misses about **7 in 10** no-shows (272 of 381). Is that a **model** problem or a **threshold** problem? *(Tomorrow's notebook is about exactly this.)*

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 9 · The tree that learned by heart

The decision tree scores **1.000** on training data and much less on test: 🚩 red flag (Day 3). It kept splitting until every leaf was pure (only "came" or only "no-show"): it memorised the training data.

In [ ]:
# ✏️ Your turn: limit the depth (Day 3) and compare. Try 4.
pruned_pipe, pruned_row = evaluate("Decision tree (depth 4)", DecisionTreeClassifier(max_depth=___, random_state=42))
results = pd.concat([results, pd.DataFrame([pruned_row]).set_index("model").round(3)])
results

In [ ]:
# ✅ Check
assert pruned_pipe.named_steps["model"].get_depth() == 4
print(f"✅ Train {pruned_row['train accuracy']:.3f} vs test {pruned_row['test accuracy']:.3f}: the gap is gone, and test AUC jumped from "
      f"{results.loc['Decision tree', 'test AUC']:.3f} to {pruned_row['test AUC']:.3f}.")

In [ ]:
# ▶️ Run: the top of the pruned tree, the part you could show Dr. Okafor
names = pruned_pipe.named_steps["prep"].get_feature_names_out()
plt.figure(figsize=(14, 5))
plot_tree(pruned_pipe.named_steps["model"], max_depth=2, feature_names=list(names),
          class_names=["came", "no-show"], filled=True, impurity=False, proportion=True, fontsize=9)
plt.show()

> 💡 The tree splits on **scaled** numbers (e.g. `num__lead_days <= -0.2`), because the recipe scales first. Trees don't need scaling (Day 3); it just makes the labels harder to read.

**Predict, then run (Day 2):** what happens to **KNN** if we remove the scaler? A) nothing B) a bit worse C) clearly worse

In [ ]:
# ▶️ Run: KNN with the same recipe but WITHOUT scaling
prep_unscaled = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)])
knn_unscaled = Pipeline([("prep", prep_unscaled), ("model", KNeighborsClassifier(n_neighbors=15))]).fit(X_train, y_train)
print("KNN test AUC, scaled:  ", results.loc["KNN (k=15)", "test AUC"])
print("KNN test AUC, unscaled:", round(roc_auc_score(y_test, knn_unscaled.predict_proba(X_test)[:, 1]), 3))

Answer: **C**, test AUC falls from 0.74 to 0.70. Without scaling, the columns with big numbers (e.g. **`lead_days` (0–120) and `age` (18–92)**) dominate the distance, and a 1-point difference in `prior_no_shows`, one of the best signals, barely counts.

🧠 **Think:** Dr. Okafor wants to understand why a patient is flagged. Which model would you show her, and what would you say about its **performance cost**?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 10 · What does the model rely on?

In [ ]:
# ▶️ Run: logistic regression's weights
# Numeric features were scaled to the same ruler, so their weights compare fairly. The one-hot columns are 0/1, not scaled:
# compare those with care. A weight shows direction and rough strength INSIDE the model, not a cause.
lr = fitted["Logistic regression"]
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
top = weights.reindex(weights.abs().sort_values(ascending=False).index).head(10)
print(top.round(2))
top.sort_values().plot.barh(color=np.where(top.sort_values() > 0, "#d8553b", "#3b7dd8"), figsize=(8, 4),
                            title="Top 10 weights (red = more likely no-show)")
plt.show()

In [ ]:
# ✏️ Your turn: which TWO features push the no-show score UP the most? Copy their names from the output above, as strings (in quotes).
strongest_up = {___, ___}

In [ ]:
# ✅ Check
assert strongest_up == set(weights.sort_values().index[-2:]), "Look for the two largest POSITIVE weights"
print("✅ Correct: past no-shows and booking far ahead. The second one is the same story as the bar chart in Step 4.")
print("🏁 M3 reached: you know which model you'd pick and what it relies on.")

🧠 **Think:** Pick **one** of the strong features. Write a product hypothesis for **why** it might matter, and one product change (not a model) that could reduce no-shows for those patients.

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

🧠 **Think (bring this to the stand-up):** Which model do you take into tomorrow? One sentence each on **performance**, **explainability** and **risk**.

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

### 🎉 Done for today
- **M1** honest table · **M2** three models vs a baseline · **M3** a model you can defend.
- **Tomorrow (notebook 2):** scores instead of yes/no, the right rule for each action, money, fairness, and your recommendation for Lena.